# Week 02 — MLP vs. Linear Baseline


## Objective
Build a two-hidden-layer multilayer perceptron (MLP) for Fashion-MNIST and compare it with the Week 1 linear classifier using the exact same training and validation split.

## 1. Setup


In [18]:
import platform
from pathlib import Path

import numpy as np
import torch
from torchvision import datasets, transforms
from torch.utils.data import Subset, DataLoader

# Q1/notebooks/ is the current notebook folder,
# so parent points to Q1/
Q1_DIR = Path.cwd().parent

DATA_DIR = Q1_DIR / "data"
SPLIT_DIR = DATA_DIR / "split_indices"
RESULTS_DIR = Q1_DIR / "results" / "raw"
FIGURES_DIR = Q1_DIR / "figures"
CONFIG_DIR = Q1_DIR / "configs"

# Use Apple's MPS accelerator when available.
# Otherwise, use the CPU.
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

print("Python version:", platform.python_version())
print("PyTorch version:", torch.__version__)
print("Device:", device)

Python version: 3.14.0
PyTorch version: 2.14.0
Device: mps


## 2. Load Fashion-MNIST

In [19]:
transform = transforms.ToTensor()
full_train_dataset = datasets.FashionMNIST(
    root=DATA_DIR,
    train=True,
    download=True,
    transform=transform
)

print("Full Fashion-MNIST training set:", len(full_train_dataset))

Full Fashion-MNIST training set: 60000


## 3. Load Week 1 Train/Validation Split

In [20]:
train_indices = np.load(SPLIT_DIR / "train_indices.npy")
val_indices = np.load(SPLIT_DIR / "val_indices.npy")

print("Training indices:", len(train_indices))
print("Validation indices:", len(val_indices))

Training indices: 10000
Validation indices: 5000


In [21]:
#These two commands turn both of the inidicies into datasets that can be used for training and validation. 
train_dataset = Subset(
    full_train_dataset,
    train_indices
)

val_dataset = Subset(
    full_train_dataset,
    val_indices
)

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))

Training dataset: 10000
Validation dataset: 5000


## 4. DataLoaders

In [22]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

## 5. Week 1 Linear Baseline

The Week 1 linear classifier is recreated using the same Fashion-MNIST
training and validation split. This provides a baseline that can be compared
directly with the Week 2 multilayer perceptron.

In [23]:
import torch.nn as nn
import torch.optim as optim
# Set a seed so the model's initial weights are reproducible.
torch.manual_seed(42)
# This is the same linear classifier architecture used in Week 1.
class LinearClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        # Converts each 28 x 28 Fashion-MNIST image
        # into a vector containing 784 pixel values.
        self.flatten = nn.Flatten()
        # Takes the 784 pixel values and produces
        # 10 logits, one for each Fashion-MNIST class.
        self.linear = nn.Linear(28 * 28, 10)
    def forward(self, x):
        # Shape changes from:
        # [batch_size, 1, 28, 28]
        # to:
        # [batch_size, 784]
        x = self.flatten(x)
        # Shape changes from:
        # [batch_size, 784]
        # to:
        # [batch_size, 10]
        x = self.linear(x)
        # Return the 10 logits for every image.
        return x
# Create the actual model and move it to MPS or CPU.
baseline_model = LinearClassifier().to(device)
# CrossEntropyLoss compares the model's 10 logits
# against the correct Fashion-MNIST class.
baseline_loss_fn = nn.CrossEntropyLoss()
# Use the exact same optimizer and learning rate as Week 1.
baseline_optimizer = optim.SGD(
    baseline_model.parameters(),
    lr=0.01
)
# Keep the same 5-epoch limit as Week 1.
EPOCHS = 5
print(baseline_model)

LinearClassifier(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear): Linear(in_features=784, out_features=10, bias=True)
)


In [24]:
# Take one batch from the training DataLoader.
images, labels = next(iter(train_loader))

# Move the images to the same device as the model.
images = images.to(device)

# Pass the images through the linear classifier.
outputs = baseline_model(images)

print("Input shape:", images.shape)
print("Output shape:", outputs.shape)

Input shape: torch.Size([32, 1, 28, 28])
Output shape: torch.Size([32, 10])


WEEK 1 Linear Classifer Traning

In [25]:
baseline_train_losses = []
baseline_val_losses = []
baseline_val_accuracies = []

EPOCHS = 5

In [32]:

torch.manual_seed(42)

baseline_model = LinearClassifier().to(device)
baseline_loss_fn = nn.CrossEntropyLoss()
baseline_optimizer = optim.SGD(
    baseline_model.parameters(),
    lr=0.01
)
EPOCHS = 5
baseline_train_losses = []
baseline_val_losses = []
baseline_val_accuracies = []
for epoch in range(EPOCHS):
    # Put the model into training mode.
    baseline_model.train()
    # Start the total training loss at zero for this epoch.
    total_train_loss = 0.0
    # Go through the training dataset one batch at a time.
    for images, labels in train_loader:
        # Move the images and labels to the same device as the model.
        images = images.to(device)
        labels = labels.to(device)
        # Clear gradients left over from the previous batch.
        baseline_optimizer.zero_grad()
        # Forward pass:
        # send the images through the linear classifier.
        outputs = baseline_model(images)
        # Compare the model's predictions with the correct labels.
        loss = baseline_loss_fn(
            outputs,
            labels
        )
        # Backpropagation:
        # calculate gradients for all trainable weights and biases.
        loss.backward()
        # Update the model's weights and biases using SGD.
        baseline_optimizer.step()
        # Add the loss from this batch to the running total.
        total_train_loss += loss.item()
    # Calculate the average training loss across all batches.
    average_train_loss = (
        total_train_loss / len(train_loader)
    )
    # Save this epoch's training loss.
    baseline_train_losses.append(
        average_train_loss
    )

    # Put the model into evaluation mode.
    baseline_model.eval()

    # Start validation totals at zero.
    total_val_loss = 0.0
    correct = 0
    total = 0
    # Gradients are not needed during validation because
    # we are only measuring performance, not training.
    with torch.no_grad():

        # Go through the validation set one batch at a time.
        for images, labels in val_loader:

            # Move the validation data to the same device as the model.
            images = images.to(device)
            labels = labels.to(device)
            # Make predictions on the validation images.
            outputs = baseline_model(images)
            # Calculate validation loss.
            loss = baseline_loss_fn(
                outputs,
                labels
            )
            # Add this batch's validation loss.
            total_val_loss += loss.item()
            # Find the predicted class.
            # argmax chooses the index of the largest logit.
            predictions = outputs.argmax(
                dim=1
            )
            # Count how many predictions are correct.
            correct += (
                predictions == labels
            ).sum().item()
            # Count the total number of validation examples.
            total += labels.size(0)
    # Calculate average validation loss for this epoch.
    average_val_loss = (
        total_val_loss / len(val_loader)
    )
    # Calculate validation accuracy.
    val_accuracy = (
        correct / total
    )
    # Save this epoch's validation results.
    baseline_val_losses.append(
        average_val_loss
    )
    baseline_val_accuracies.append(
        val_accuracy
    )
    # Print the results for this epoch.
    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train Loss: {average_train_loss:.4f} | "
        f"Val Loss: {average_val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy * 100:.2f}%"
    )

Epoch 1/5 | Train Loss: 1.2902 | Val Loss: 0.9640 | Val Accuracy: 69.26%
Epoch 2/5 | Train Loss: 0.8622 | Val Loss: 0.8067 | Val Accuracy: 73.54%
Epoch 3/5 | Train Loss: 0.7556 | Val Loss: 0.7518 | Val Accuracy: 75.54%
Epoch 4/5 | Train Loss: 0.6982 | Val Loss: 0.6915 | Val Accuracy: 78.16%
Epoch 5/5 | Train Loss: 0.6602 | Val Loss: 0.6666 | Val Accuracy: 78.06%


Epoch 5/5 | Train Loss: 0.6607 | Val Loss: 0.6703 | Val Accuracy: 78.18%


## 6. Week 2 MLP Architecture

In [28]:
class MLPClassifier(nn.Module):

    def __init__(self):
        super().__init__()

        # Convert each 28 x 28 image into 784 pixel values.
        self.flatten = nn.Flatten()

        # Build the neural network.
        self.network = nn.Sequential(

            # First hidden layer:
            # 784 input pixel values -> 128 hidden neurons
            nn.Linear(28 * 28, 128),

            # Apply a nonlinear activation function.
            nn.ReLU(),

            # Second hidden layer:
            # 128 values -> 64 hidden neurons
            nn.Linear(128, 64),

            # Another nonlinear activation function.
            nn.ReLU(),

            # Output layer:
            # 64 hidden values -> 10 class logits
            nn.Linear(64, 10)
        )

    def forward(self, x):

        # [batch, 1, 28, 28]
        # becomes
        # [batch, 784]
        x = self.flatten(x)

        # Pass the flattened image through
        # all three linear layers and both ReLUs.
        x = self.network(x)

        # Return 10 logits for each image.
        return x


mlp_model = MLPClassifier().to(device)

print(mlp_model)

MLPClassifier(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (network): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)


## 7. Training


In [33]:
# CrossEntropyLoss compares the model's 10 output logits
# to the correct Fashion-MNIST labels.
mlp_loss_fn = nn.CrossEntropyLoss()


# Use the same optimizer and learning rate as the linear baseline
# so that the comparison between the two models is more controlled.
mlp_optimizer = optim.SGD(
    mlp_model.parameters(),
    lr=0.01
)


# Store the results from every epoch.
# We will use these later to make learning curves.
mlp_train_losses = []
mlp_val_losses = []
mlp_val_accuracies = []


# Use the same number of epochs as the baseline.
EPOCHS = 5

## 8. Validation

## 9. Learning Curves

## 10. Linear vs. MLP Comparison

## 11. Example Prediction


## 12. Failure Example

## 14. Reflection
